In [1]:
# Import necessary libraries
import json
import os
from typing import Any, Dict

import boto3
import requests
from dotenv import load_dotenv

load_dotenv()
print("We are up and running!")

We are up and running!


AWS credentials are picked up automatically from the environment (instance profile, env vars, or `~/.aws/credentials`).

In [2]:
MODEL_ID = "eu.amazon.nova-lite-v1:0"
REGION = "eu-west-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Bedrock client ready — model: {MODEL_ID}")

Bedrock client ready — model: eu.amazon.nova-lite-v1:0


In [3]:
PROMPT = "What is the current price of Bitcoin?"

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=[
        {"role": "user", "content": [{"text": PROMPT}]}
    ],
)

print(response["output"]["message"]["content"][0]["text"])

I'm unable to provide real-time data as my current information is up to 2022 and I do not have access to live updates. However, you can easily find the current price of Bitcoin by checking financial news websites, cryptocurrency exchanges like Coinbase, Binance, or Kraken, or by using financial apps that track cryptocurrency prices. Simply search for "current price of Bitcoin" in your preferred search engine for the most up-to-date information. Please note that the price of Bitcoin can be highly volatile and may change frequently.


In [4]:
# Inspect the full raw response from Bedrock
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "52234835-5ed8-4fd1-bbe1-c63d763567e6",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Sun, 08 Mar 2026 14:47:18 GMT",
      "content-type": "application/json",
      "content-length": "741",
      "connection": "keep-alive",
      "x-amzn-requestid": "52234835-5ed8-4fd1-bbe1-c63d763567e6"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "I'm unable to provide real-time data as my current information is up to 2022 and I do not have access to live updates. However, you can easily find the current price of Bitcoin by checking financial news websites, cryptocurrency exchanges like Coinbase, Binance, or Kraken, or by using financial apps that track cryptocurrency prices. Simply search for \"current price of Bitcoin\" in your preferred search engine for the most up-to-date information. Please note that the price of Bitcoin can be highly volati

In [5]:
url = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"
response = requests.get(url)
data = response.json()
print(data)

{'symbol': 'BTCUSDT', 'price': '67122.67000000'}


In [6]:
# Define the function
def get_crypto_price(symbol: str) -> float:
    """
    Get the current price of a cryptocurrency from Binance API
    """
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    data = response.json()
    return float(data["price"])

In [7]:
price = get_crypto_price("BTCUSDT")
print(f"BTC Price in USDT: {price}")

BTC Price in USDT: 67122.67


In [8]:
# Tool definition in Bedrock Converse format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_crypto_price",
                "description": "Get cryptocurrency price in USDT from Binance",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The cryptocurrency trading pair symbol "
                                    "(e.g., BTCUSDT, ETHUSDT). "
                                    "The symbol for Bitcoin is BTCUSDT. "
                                    "The symbol for Ethereum is ETHUSDT."
                                ),
                            }
                        },
                        "required": ["symbol"],
                    }
                },
            }
        }
    ]
}

In [9]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "8e07bb15-d2b2-4d0a-8d28-05235367ae47",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Sun, 08 Mar 2026 14:47:20 GMT",
      "content-type": "application/json",
      "content-length": "569",
      "connection": "keep-alive",
      "x-amzn-requestid": "8e07bb15-d2b2-4d0a-8d28-05235367ae47"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>The User has asked for the current price of Bitcoin. I have the tool \"get_crypto_price\" which can fetch the current price of a cryptocurrency from Binance. I need to use this tool with the symbol for Bitcoin, which is BTCUSDT.</thinking>\n"
        },
        {
          "toolUse": {
            "toolUseId": "tooluse_pSvZmrw8biUH6r66GrhDJp",
            "name": "get_crypto_price",
            "input": {
              "symbol": "BTCUSDT"
            }
          }
        }
      ]
    }
  },
 

In [10]:
# Extract the tool call from the response
assistant_message = response["output"]["message"]
tool_use_block = next(
    block for block in assistant_message["content"] if "toolUse" in block
)

tool_use_id = tool_use_block["toolUse"]["toolUseId"]
tool_name = tool_use_block["toolUse"]["name"]
tool_input = tool_use_block["toolUse"]["input"]

print(f"Tool requested: {tool_name}")
print(f"Tool input:     {tool_input}")
print(f"Tool use ID:    {tool_use_id}")

Tool requested: get_crypto_price
Tool input:     {'symbol': 'BTCUSDT'}
Tool use ID:    tooluse_pSvZmrw8biUH6r66GrhDJp


In [11]:
# Manually call the function with the arguments the model provided
price = get_crypto_price(**tool_input)
print(f"Result from function: {price}")

Result from function: 67119.36


In [12]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "04dde457-ccf6-400a-bc73-4a652f084964",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Sun, 08 Mar 2026 14:47:21 GMT",
      "content-type": "application/json",
      "content-length": "493",
      "connection": "keep-alive",
      "x-amzn-requestid": "04dde457-ccf6-400a-bc73-4a652f084964"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>I have received the result from the tool \"get_crypto_price\" for Bitcoin (BTCUSDT). The current price of Bitcoin is $67,119.36.</thinking>\n\n\n\nThe current price of Bitcoin (BTC) is $67,119.36. Please note that cryptocurrency prices are highly volatile and can change rapidly."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 558,
    "outputTokens": 86,
    "totalTokens": 644
  },
  "metrics": {
    "latencyMs": 814
  }
}


In [13]:
import re

text = final_response["output"]["message"]["content"][0]["text"]
clean_text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

print(clean_text)

The current price of Bitcoin (BTC) is $67,119.36. Please note that cryptocurrency prices are highly volatile and can change rapidly.
